<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install -U duckdb

import duckdb
from google.colab import userdata

# Read the token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Start DuckDB
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Give DuckDB access to the gated Hugging Face dataset.
# The actual token is NOT written in the notebook.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{safe_token}'
);
""")

# We intentionally work only on a middle month.
MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("DuckDB setup complete.")
print("Development month: 2026-03")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 58.5 MB/s eta 0:00:00
DuckDB setup complete.
Development month: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
one row for each report date × pseudonymized client × pseudonymized content item.

In the warehouse, the pseudonymized identifiers are stored as client_hash_id and content_hash_id. I verify the grain by grouping on report_date, client_hash_id, and content_hash_id and checking whether any combination appears more than once.

If the query returns no rows, the documented daily grain holds for my March 2026 slice.

In [ ]:
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{MARCH_PATH}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

grain_check = con.sql(grain_query).df()

display(grain_check)

if grain_check.empty:
    print(
        "PASS: No duplicate report_date × client_hash_id × content_hash_id "
        "groups were found. The documented daily grain holds."
    )
else:
    print(
        "FAIL: Duplicate daily grain combinations were found. "
        "The contract needs to be investigated."
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


PASS: No duplicate report_date × client_hash_id × content_hash_id groups were found. The documented daily grain holds.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

March 2026 partition of fact_content_daily_performance.

I verify the number of rows in this slice and the minimum and maximum report_date. This confirms both the size of the slice and that I am using the intended month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
count_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{MARCH_PATH}');
"""

count_window = con.sql(count_window_query).df()

display(count_window)

row = count_window.iloc[0]

print(
    f"The March 2026 slice contains {int(row['row_count']):,} rows "
    f"and covers {row['first_date']} through {row['last_date']}."
)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 rows and covers 2026-03-01 00:00:00 through 2026-03-31 00:00:00.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Not every daily row has valid GA4 measurements. Some rows can contain zero-filled GA4 values because analytics data was not yet available for that client.

I therefore check ga4_data_available explicitly and filter with IS TRUE. This tells me how many rows in my March 2026 slice can safely be treated as having measured GA4 information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS FALSE
    ) AS ga4_unavailable_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NULL
    ) AS ga4_unknown_rows,

    ROUND(
        100.0 *
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        / COUNT(*),
        2
    ) AS pct_survive

FROM read_parquet('{MARCH_PATH}');
"""

availability = con.sql(availability_query).df()

display(availability)

row = availability.iloc[0]

print(
    f"{int(row['ga4_available_rows']):,} of "
    f"{int(row['total_rows']):,} March rows survive "
    f"`ga4_data_available IS TRUE` "
    f"({row['pct_survive']:.2f}%)."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_unavailable_rows,ga4_unknown_rows,pct_survive
0,9841378,413966,6408671,3018741,4.21


413,966 of 9,841,378 March rows survive `ga4_data_available IS TRUE` (4.21%).


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One important limitation of my March 2026 slice is limited GA4 availability.

Only 413,966 of 9,841,378 rows, or 4.21%, have ga4_data_available IS TRUE. This means most rows in this month do not have confirmed GA4 measurements.

Therefore, GA4 metrics such as sessions or engagement should not be treated as universally available features. A zero value may mean the measurement was unavailable rather than that no activity occurred.

Because of this, I will rely mainly on GSC-based features for the main feature frame and treat GA4-based analysis as applying only to the smaller subset where availability is explicitly true.

This also means my conclusions are decision-support and directional, not a complete description of all content behavior.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Reuse the result from Verification 3
limit_row = availability.iloc[0]

total_rows = int(limit_row["total_rows"])
ga4_available_rows = int(limit_row["ga4_available_rows"])
pct_survive = float(limit_row["pct_survive"])

print("Named limitation: Limited GA4 availability")
print(f"Total March rows: {total_rows:,}")
print(f"Rows with ga4_data_available IS TRUE: {ga4_available_rows:,}")
print(f"Percentage with measured GA4 availability: {pct_survive:.2f}%")

if pct_survive < 50:
    print(
        "\nInterpretation: GA4-based features are available for only a minority "
        "of this slice, so they should not be treated as universally observed."
    )

Named limitation: Limited GA4 availability
Total March rows: 9,841,378
Rows with ga4_data_available IS TRUE: 413,966
Percentage with measured GA4 availability: 4.21%

Interpretation: GA4-based features are available for only a minority of this slice, so they should not be treated as universally observed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.